In [0]:
%load_ext autoreload
%autoreload 2
# autoreload: if you edit a file in src/, notebooks pick up the change without a restart

from src import config
config.configure_adls(spark)          # one line replaces the 10-line login cell

RAW_PATH = config.RAW_PATH
print("Config loaded. Silver path:", config.SILVER_PATH)

In [0]:
from pyspark.sql import functions as F

raw = (spark.read
       .option("header", True)
       .option("inferSchema", False)  # keep everything as strings for profiling
       .option("nullValue", "null")   # treat the literal text "null" as a real null
       .option("escape", '"')         # handle """CNR...""" quoting correctly
       .csv(RAW_PATH))

print("Rows:", raw.count(), "| Columns:", len(raw.columns))
display(raw.limit(5))

In [0]:
null_counts = raw.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in raw.columns])
display(null_counts)

In [0]:
display(raw.groupBy("Booking Status").count().orderBy(F.desc("count")))

In [0]:
outcome_cols = ["Avg VTAT", "Avg CTAT", "Booking Value", "Ride Distance",
                "Driver Ratings", "Customer Rating", "Payment Method"]
display(raw.groupBy("Booking Status").agg(
    F.count("*").alias("rows"),
    *[F.round(F.avg(F.col(c).isNotNull().cast("int")) * 100, 1).alias(f"{c} %filled")
      for c in outcome_cols]
))

In [0]:
display(raw.groupBy("Booking Status",
                    "Cancelled Rides by Customer",
                    "Cancelled Rides by Driver",
                    "Incomplete Rides").count().orderBy("Booking Status"))

# A reason should be present exactly when its flag is set. Count violations:
pairs = [("Cancelled Rides by Customer", "Reason for cancelling by Customer"),
         ("Cancelled Rides by Driver", "Driver Cancellation Reason"),
         ("Incomplete Rides", "Incomplete Rides Reason")]
for flag, reason in pairs:
    bad = raw.filter(F.col(flag).isNull() != F.col(reason).isNull()).count()
    print(f"{flag} vs reason mismatches: {bad}")

In [0]:
display(raw.groupBy("Booking Status").agg(
    F.round(F.avg(F.col("Avg VTAT").cast("double")), 2).alias("avg_VTAT"),
    F.round(F.avg(F.col("Avg CTAT").cast("double")), 2).alias("avg_CTAT"),
    F.sum(F.col("Avg VTAT").isNotNull().cast("int")).alias("VTAT_present"),
    F.sum(F.col("Avg CTAT").isNotNull().cast("int")).alias("CTAT_present"),
))

In [0]:
fmt = raw.select(
    F.sum(F.col("Date").rlike(r"^\d{4}-\d{2}-\d{2}$").cast("int")).alias("yyyy-MM-dd"),
    F.sum(F.col("Date").rlike(r"^\d{2}-\d{2}-\d{4}$").cast("int")).alias("dd-MM-yyyy (PDF)"),
    F.sum(F.col("Time").rlike(r"^\d{2}:\d{2}:\d{2}$").cast("int")).alias("valid HH:mm:ss"),
    F.min("Date").alias("min_date"), F.max("Date").alias("max_date"),
    F.countDistinct("Date").alias("distinct_dates"))
display(fmt)

In [0]:
display(raw.select("Booking ID", "Customer ID").limit(3))   # shows the embedded quotes

dups = raw.groupBy("Booking ID").count().filter("count > 1")
print("Duplicate Booking IDs:", dups.count(),
      "| extra rows:", dups.agg(F.sum(F.col("count") - 1)).first()[0])
print("Fully identical duplicate rows:", raw.count() - raw.distinct().count())

In [0]:
for c in ["Vehicle Type", "Payment Method", "Reason for cancelling by Customer",
          "Driver Cancellation Reason", "Incomplete Rides Reason"]:
    print(f"\n== {c}")
    raw.groupBy(c).count().orderBy(F.desc("count")).show(truncate=False)

In [0]:
nums = ["Avg VTAT", "Avg CTAT", "Booking Value", "Ride Distance", "Driver Ratings", "Customer Rating"]
display(raw.select([F.col(c).cast("double").alias(c) for c in nums]).summary("min", "25%", "50%", "75%", "max", "mean"))

In [0]:
daily = raw.groupBy("Date").count()
display(daily.agg(F.min("count").alias("min_per_day"), F.max("count").alias("max_per_day"),
                  F.round(F.avg("count"), 1).alias("avg_per_day")))
display(daily.orderBy("Date"))   # tip: switch this output to a line chart for a report screenshot